## 初期設定

In [1]:
# 更新履歴
## 2026/01/23 新peakfitに適合するようにしました
## 2026/01/23 imshowをlogで表示するようにしました
## 2026/01/22 ピークフィットの結果をcsvとして保存できるようにしました
## 2026/01/05 save_tiffの追加
## 2025/12/29 disp_cakingの作成
## 2025/12/17 plotlyを組み込み
## 2025/12/17 作成

# 基本モジュール
import sys, os
from IPython.display import display, HTML, clear_output, update_display, Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pyFAI
import pandas as pd
import h5py
import plotly.express as px
import plotly.graph_objects as go
from tqdm import tqdm
import PIL.Image as im
import json
from typing import Union, Optional

# fpdファイル取扱い用
import threading
import concurrent.futures as confu

# 自作モジュール
sys.path.append(r"C:\Users\okaza\pythonenv")
from modules.Mytools.handle_ipynb import save_pickle, load_pickle, export_html, ask_openfilename, ask_savefilename
from modules.Mytools.Tools import h5_tree, dict_tree, his2array, simple_progress_bar
from modules.Mytools.PseudoVoigt import peakfit, pseudoVoigt
import modules.Mytools.Settings
sys.path.append(os.getcwd())

# 初期パラメーター
cachedir = os.path.join(os.getcwd(), ".cache")
os.makedirs(cachedir, exist_ok=True)

## 目的

SPring-8 BL10XUで取得したhisデータをhdfファイルに保存する。

## 初期化

In [2]:
Experiment = "UODE42_0004"

In [3]:
# 格納用データの作成
config = dict()
config["name"] = Experiment

## ファイルリストの作成

In [4]:
# ディレクトリ名を指定
dir = r"D:\DATA\SPring-8-2026-April\UODE42\FPD"

# ヘッダーとフッターを指定
header = "UODE42_4_"
footer = ".his"

In [5]:
# ファイルリストの格納
def get_filelist(dir: str,
                 header: str,
                 footer: str
                 ) -> list:
    
    print("="*70)
    print("Inputs:")
    print("\tdir:    " + dir)
    print("\theader: " + header)
    print("\tfooter: " + footer)
    print("="*70)

    ## ヘッダーとフッターを含むファイル名を取得
    flist = list()
    for __ in os.listdir(dir):
        if not header in __:
            continue
        if not footer in __:
            continue
        flist.append(__)

    ## ソート
    flist.sort(key = (lambda x: int(x.replace(header, "").replace(footer, ""))))
    filenames = list(map(lambda x: os.path.join(dir, x), flist))

    ## 表示
    for f in filenames:
        print(f)
    
    return filenames

## 格納
config["filelist"] = get_filelist(dir, header, footer)
del get_filelist

Inputs:
	dir:    D:\DATA\SPring-8-2026-April\UODE42\FPD
	header: UODE42_4_
	footer: .his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_0.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_1.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_2.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_3.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_4.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_5.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_6.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_7.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_8.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_9.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_10.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_11.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_12.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_13.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_14.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UODE42_4_15.his
D:\DATA\SPring-8-2026-April\UODE42\FPD\UO

## 一次元化

poniファイル（校正用ファイル）を用いて2次元画像を1次元化します。  
poniファイルを作成する際は、pyFAI-calib2を用いると簡単にできます。  
なお、hisファイルをtiffファイルに変換する際は、his2tiff.pyが使えます。

In [6]:
# 校正用ファイル
poni = r"D:\DATA\SPring-8-2026-April\CeO2_20260410\pyFAI-calib2_EH2_20260602_YusukeOkazaki.poni"

# step
step = 0.005

# 2thetaの範囲
radial_range = (3,30)

In [7]:
# 一気に1次元化する
def integrate_1d(poni: str,
                 step: float,
                 radial_range: tuple[float, float]) -> str:
    
    npt_rad = int((radial_range[1] - radial_range[0])/step)
    
    # hdfファイル初期化
    integratedhdf = os.path.join(cachedir, config["name"] + "_integrate1d.hdf")
    with h5py.File(integratedhdf, mode = "w") as f:
        f.create_dataset(
            name = "rad",
            shape = (npt_rad,),
            dtype = np.float32,
        )
        g = f.create_group(
            name = "integrated"
        )
        for i in tqdm(range(len(config["filelist"]))):
            g.create_dataset(
                name = "frame = {}".format(i),
                shape = (npt_rad,),
                dtype = np.float32
            )
    
    ai = pyFAI.load(poni)
    lock = threading.Lock()
    
    def func(i):

        # データ読み込み
        d = his2array(config["filelist"][i])

        # 積算する
        radial, intensity = ai.integrate1d(
            data = d,
            npt = npt_rad,
            radial_range=radial_range,
            unit = "2th_deg",
            # method = "ocl"
        )

        return i, intensity, radial

    with confu.ThreadPoolExecutor(max_workers=os.cpu_count()) as tpe:

        futures = [tpe.submit(func, i) for i in range(len(config["filelist"]))]

        for i, future in enumerate(confu.as_completed(futures)):
            j, intensity, radial = future.result()
            with lock:
                with h5py.File(integratedhdf, mode = "r+") as f:
                    if not i:
                        f["rad"][:] = radial # type: ignore
                    f["integrated/frame = {}".format(j)][:] = intensity # type: ignore
                simple_progress_bar(i+1, len(config["filelist"]))

    # 出力
    print("\nIntegration completed.")
    with h5py.File(integratedhdf, mode = "r") as f:
        h5_tree(f)
    return integratedhdf
hdf_1d = integrate_1d(poni, step, radial_range)
del integrate_1d 

100%|██████████| 68/68 [00:00<00:00, 28667.47it/s]


Progress: [■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■] 100% (68/68) 

Integration completed.
<HDF5 file "UODE42_0004_integrate1d.hdf" (mode r)>
├── integrated
│   ├── frame = 0 ((5400,), float32)
│   ├── frame = 1 ((5400,), float32)
│   ├── frame = 10 ((5400,), float32)
│   ├── frame = 11 ((5400,), float32)
│   ├── frame = 12 ((5400,), float32)
│   ├── frame = 13 ((5400,), float32)
│   ├── frame = 14 ((5400,), float32)
│   ├── frame = 15 ((5400,), float32)
│   ├── frame = 16 ((5400,), float32)
│   ├── frame = 17 ((5400,), float32)
│   ├── frame = 18 ((5400,), float32)
│   ├── frame = 19 ((5400,), float32)
│   ├── frame = 2 ((5400,), float32)
│   ├── frame = 20 ((5400,), float32)
│   ├── frame = 21 ((5400,), float32)
│   ├── frame = 22 ((5400,), float32)
│   ├── frame = 23 ((5400,), float32)
│   ├── frame = 24 ((5400,), float32)
│   ├── frame = 25 ((5400,), float32)
│   ├── frame = 26 ((5400,), float32)
│   ├── frame = 27 ((5400,), float32)
│   ├── frame = 28 ((5400,), float32)
│   ├── fr

結果を確認する

In [8]:
# ファイルを読み込んで、plotlyで出力する
def get_data(hdf: Union[str, list[str]],
             ) -> tuple[np.ndarray, np.ndarray]:
    
    print("="*70)
    print("hdf:")
    print(json.dumps(hdf) if type(hdf) == list else hdf)
    print("="*70)

    # データ読み込み
    ity = []
    if type(hdf) == str:
        with h5py.File(hdf, mode = "r") as f:
            tth = np.array(f["rad"][()]) # type: ignore
            for i in range(len(list(f["integrated"].keys()))): # type: ignore
                ity.append(f["integrated/frame = {}".format(i)][()]) # type: ignore
    elif type(hdf) == list:
        for file in hdf:
            with h5py.File(file, mode = "r") as f:
                tth = np.array(f["rad"][()]) # type: ignore
                for i in range(len(list(f["integrated"].keys()))): # type: ignore
                    ity.append(f["integrated/frame = {}".format(i)][()]) # type: ignore
            print(os.path.basename(file) + ": {} frames".format(len(ity)))
    ity = np.vstack(ity)

    # figure作成
    if True:

        # インスタント化
        fig = go.Figure()

        # 一枚目作成
        raw_anim = go.Scatter(
            x = tth,
            y = ity[0],
            mode = "lines",
            line = dict(
                color = "#000000",
                width = 1,
            )
        )
        fig.add_trace(raw_anim)

        # アニメーションフレームを作成
        frame_list = [
            go.Frame(
                data = [
                    go.Scatter(
                        y = ity[k]
                    )
                ],
                name = str(k),
            ) for k in range(ity.shape[0])
        ]
        fig.frames = frame_list

        # フレーム変更用のボタンとスライダー
        updatemenus_frame = dict(
            type='buttons',
            direction = 'left', # 横並びの場合 'left', 縦並びの場合 'down'
            showactive=False,
            active = -1,
            y=0,
            x=0,
            xanchor='left',
            yanchor='top',
            pad=dict(t=65, r=0),
            buttons=[
                dict(
                    label='▶︎',
                    method='animate',
                    args=[
                        None,
                        dict(
                            frame=dict(
                                duration=200,
                                redraw=True
                            ),
                            fromcurrent=True,
                            mode='immediate',
                            transition=dict(duration=0)
                        )
                    ]
                ),
                dict(
                    label='⏸',
                    method='animate',
                    args=[
                        [None],
                        dict(
                            frame=dict(
                                duration=0,
                                redraw=False
                            ),
                            mode='immediate',
                            transition=dict(duration=0)
                        )
                    ]
                )
            ],
            visible = True
        )
        slider = dict(
            steps=[dict(
                method='animate',
                args=[
                    [str(k)],
                    dict(
                        mode='immediate',
                        frame=dict(
                            duration=0,
                            redraw=True
                            ),
                        transition=dict(
                            duration=0
                        )
                    )
                ],
                label=str(k),
            ) for k in range(ity.shape[0])],
            ticklen = 0,
            minorticklen = 0,
            active=0,
            y=0,
            x=0, # スライダーの開始位置
            len=1, # スライダーの長さ
            xanchor='left',
            yanchor='top',
            pad=dict(
                b=10,
                t=50,
                l=100
            ),
            currentvalue=dict(
                prefix='Frame = ',
                visible=True,
                xanchor='left'
            ),
            visible = True,
        )

        fig.update_layout(
            title = dict(
                text = os.path.splitext(os.path.basename(hdf))[0] if type(hdf) == str else None,
                font = dict(
                    size = 26,
                    color = "#777777"
                ),
                x = 0.05,
                y = 0.95,
                xanchor = "left",
                yanchor = "top",
            ),
            xaxis = {
                'title': "2theta [deg.]",
                "linecolor": "#000000",
                "linewidth": 1,
                'ticks': "inside",
                "tickcolor": "#000000",
                "tickwidth": 1,
                "ticklen": 5,
                'mirror': True,
            },
            yaxis = {
                'title': "Frame",
                "linecolor": "#000000",
                "linewidth": 1,
                'ticks': "inside",
                "tickcolor": "#000000",
                "tickwidth": 1,
                "ticklen": 5,
                'mirror': True,
            },
            plot_bgcolor = '#ffffff',
            updatemenus = [
                updatemenus_frame,
            ],
            sliders = [slider],
            showlegend = False,
            legend = dict(
                bordercolor = "#000000",
                orientation = "h",
                yanchor = "bottom",
                y = 1,
                bgcolor = "rgba(255,255,255,0)",
                xanchor = "center",
                x = 0.5
            ),
            width = 800,
            height = 600,
        )

        fig.show()

    return (tth, ity)
tth, ity = get_data(hdf_1d)
del get_data

hdf:
c:\Users\okaza\Documents\Documents\fpd\create_hdfdata\create_hdfdata\.cache\UODE42_0004_integrate1d.hdf


## csv作成

In [9]:
# hdfファイルを指定
hdf_csv = hdf_1d
# hdf_csv = r"D:\DATA\SPring-8-2025-Dec\Analysis\FPD\create_hdfdata\.cache\UODE36_0024_integrate1d.hdf"

# 1枚をcsvにするか、複数のフレームの平均をcsvにするか
flag_1shot = True
# flag_1shot = False

# 1枚の場合、フレームを指定
frame = 0

# 複数フレームの場合、レンジを指定
frame_range = (51,56)


In [10]:
# PDIndexer用にcsvを保存する
def save_csv(hdf: str,
             flag_1shot: bool,
             frame: int,
             frame_range: tuple[int, int]
             ) -> str:

    ## 1枚の場合
    if flag_1shot:
        with h5py.File(hdf, mode = "r") as f:
            tth = np.array(f["rad"][()]) # type: ignore
            insty = np.array(f["integrated/frame = {}".format(frame)][()]) # type: ignore
        csvfilename = os.path.join(cachedir, config["name"] + "_{}.csv".format(frame))

    ## 複数フレーム
    else:
        with h5py.File(hdf, mode = "r") as f:
            tth = np.array(f["rad"][()]) # type: ignore
            insty = np.zeros(shape = tth.shape)
            for i in range(*frame_range):
                insty += np.array(f["integrated/frame = {}".format(i)][()]) # type: ignore
            insty /= (frame_range[1] - frame_range[0])
        csvfilename = os.path.join(cachedir, config["name"] + "_{}-{}.csv".format(*frame_range))

    ## 保存
    df = pd.DataFrame([tth, insty]).T
    df.to_csv(csvfilename, index = False, header = False)
    print(csvfilename)

    return csvfilename
csvfilename = save_csv(hdf_csv, flag_1shot, frame, frame_range)
del save_csv

c:\Users\okaza\Documents\Documents\fpd\create_hdfdata\create_hdfdata\.cache\UODE42_0004_0.csv


## Azumuthal方向切り開き

In [ ]:
# 切り開きステップ
npt_azim = 512
npt_azim = 2048

# 切り開き範囲
radial_range = (13.55, 14)

# 表示するのはどちらか（matplotlibはいずれにせよ保存されます）
kind = "plotly"
# kind = "matplotlib"

# peakをトラッキングする
peak_tracking = True
# peak_tracking = False

# peak情報が書かれたcsv
peak_csv = r"D:\DATA\SPring-8-2025-Dec\Analysis\run2\UODE36_0024\UODE36_0024_hcp011_popts.csv"

# peakトラッキングをする場合のradial方向の積分幅 [deg.]
peak_width = 0.05

In [ ]:
# Azumuthal方向の切り開きを実施し、hdfファイルを保存する
def integrate_radial(npt_azim: int,
                     radial_range: tuple,
                     poni: str,
                     kind: str,
                     peak_tracking: bool,
                     peak_csv: str,
                     peak_width: float) -> dict:
    
    if peak_tracking:
        PeakPosi = pd.read_csv(
            peak_csv,
            usecols = [4],
        ).values
        arr_width = np.array([-0.5, 0.5]) * peak_width

    # インスタント化
    ai = pyFAI.load(poni)

    # データ格納用辞書
    d = dict()
    
    for i,f in enumerate(tqdm(i2a.filelist)):

        if peak_tracking:
            rrange = arr_width + PeakPosi[i]
        else:
            rrange = radial_range

        # データ読み込み
        hisdata = his2array(f)

        # 切り開き
        chi, insty = ai.integrate_radial(
            hisdata,
            npt = npt_azim,
            radial_unit = "2th_deg",
            method = "ocl",
            radial_range = rrange
        )

        # データ格納
        if not i:
            d["chi"] = chi
            d["data"] = []    
        d["data"].append(insty)

    # データ格納
    d["data"] = np.vstack(d["data"])
    hdffilename = os.path.join(cachedir, i2a.name + "_azim.hdf")
    with h5py.File(hdffilename, mode = "w") as f:
        f.create_dataset(
            name = "chi",
            data = d["chi"],
            dtype = d["chi"].dtype,
            shape = d["chi"].shape
        )
        f.create_dataset(
            name = "value",
            data = d["data"],
            dtype = d["data"].dtype,
            shape = d["data"].shape
        )
    print("[Save hdf]: " + hdffilename)

    if True:
        # figure作成
        fig = plt.figure()
        fig.set_size_inches(8,4.5)

        # imshowの作成
        ax = fig.add_axes(
            rect = (0.1,0.1,0.68,0.8)
        )
        cmap = ax.imshow(
            d["data"].T,
            aspect = "auto",
            extent = (
                -0.5,
                d["data"].shape[0] + 0.5,
                d["chi"][0] + (d["chi"][1]-d["chi"][0])/2,
                d["chi"][-1] + (d["chi"][1]-d["chi"][0])/2
            ),
            origin = "lower",
            norm = matplotlib.colors.LogNorm(), # type: ignore
        )
        if peak_tracking:
            title = i2a.name + " (peak tracking)"
        else:
            title = i2a.name + " (2theta: {}-{})".format(*radial_range)
        ax.set_title(title,
                    fontsize = 12,
                    loc = "left")
        ax.tick_params(direction = "out")
        ax.set_xlabel("Frame", fontsize = 14)
        ax.set_ylabel("Azimuithal angle [deg.]", fontsize = 14)

        # colorbarの作成
        cbar = fig.add_axes(
            rect = (0.8,0.1,0.02,0.8)
        )
        matplotlib.colorbar.Colorbar( # type: ignore
            mappable = cmap,
            ax = cbar,
            orientation = "vertical"
        )
        cbar.set_ylabel("Intensity [arb. unit]", fontsize = 10)
        cbar.tick_params(direction = "out")

        # 出力
        pngfilename = os.path.join(cachedir, i2a.name + "_azim.png")
        plt.savefig(pngfilename, dpi = 300)
        plt.close()
        print("[Save figure]: " + pngfilename)

    if kind == "plotly":

        # figure
        fig = go.Figure()

        # Heatmap
        heatmap = go.Heatmap(
            z = np.log(d["data"].T),
            y = d["chi"],
            colorscale = "Viridis",
            zauto = True,
            visible = True,
            name = "Heatmap",
            colorbar = {
                'title': "Intensity [arb. unit]"
            }
        )
        fig.add_trace(heatmap)


        fig.update_layout(
            title = dict(
                text = title,
                font = dict(
                    size = 18,
                    color = "#222222"
                ),
                x = 0.05,
                y = 0.9,
                xanchor = "left",
                yanchor = "top",
            ),
            xaxis = {'title': {"text": "Frame",
                            "font": {"size": 20,
                                        "color": "#222222"}}},
            yaxis = {'title': {"text": "Azumuthal angle [deg.]",
                            "font": {"size": 20,
                                        "color": "#222222"}}},
            plot_bgcolor = '#ffffff',
            showlegend = False,
            legend = dict(
                bordercolor = "#000000",
                orientation = "h",
                yanchor = "bottom",
                y = 1,
                bgcolor = "rgba(255,255,255,0)",
                xanchor = "center",
                x = 0.5
            ),
            width = 800,
            height = 600,
        )
        fig.show()

    elif kind == "matplotlib":
        display(Image(filename = pngfilename, width = 800))

    with h5py.File(hdffilename, mode = "r") as f:
        h5_tree(f)

    return d

# 例外処理
if not peak_tracking:
    peak_csv = ""
    peak_width = 0

# 演算
d_integrate_radial = integrate_radial(npt_azim, radial_range, poni, kind, peak_tracking, peak_csv, peak_width)
del integrate_radial

## Unroll

In [ ]:
# 散乱角方向のステップ
npt_step = 0.005

# 方位角方向
npt_azim = 1024
npt_azim = 2048

# 切り開き範囲
radial_range = (12,14)
radial_range = (13.55, 14)
# radial_range = (12.5,13.5)

In [ ]:
# unrollを行う
hdf_2d = i2a.integrate2D(
    poni = poni,
    npt_rad = (radial_range[-1]-radial_range[0]) / npt_step,
    npt_azim = npt_azim,
    radial_range = radial_range,
)

In [ ]:
# unrollの結果を表示する（plotly）
## http://127.0.0.1:8025/ にアクセスしてください
## python "C:\Users\okaza\Documents\Documents\fpd\dash_Integrated2D\Dash_Integrated2D.py"

## Tiff画像の出力

In [ ]:
# 保存するフレーム
frame_range = (96,108)
frame_range = (0,16)

In [ ]:
# Tiff画像の出力
def save_tiff(frame_range: tuple[int, int]) -> str:

    # 引数の出力
    print("=" * 30)
    print("Input:")
    print("\tframe_range: {}-{}".format(*frame_range))
    print("=" * 30)

    # imgファイルの生成
    hisdata = []
    for j in range(*frame_range):
        f = i2a.filelist[j]
        hisdata.append(his2array(f))
    img = np.average(np.stack(hisdata), axis = 0)

    ## 保存
    imgfilename = os.path.join(cachedir, header + "{}-{}.tif".format(*frame_range))
    im.fromarray(img).save(imgfilename)
    print("[Save tiff]: " + imgfilename)

    return imgfilename
save_tiff(frame_range)
del save_tiff